# 🗂️ Replay Buffer Frame Viewer
Scroll through every frame stored in the **replay buffer** — like a video editor timeline.

In [ ]:
import torch
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, HTML
import os, glob

# ── Style ──────────────────────────────────────────────────────────────────
display(HTML("""
<style>
  .viewer-root { background:#0d1117; border-radius:12px; padding:16px; }
  .widget-label { color:#c9d1d9 !important; font-family:'Segoe UI',sans-serif; font-size:13px; }
  .widget-hslider .ui-slider { background:#21262d !important; }
  .widget-hslider .ui-slider-handle { background:#a371f7 !important; border:none !important; border-radius:50% !important; }
  .widget-button button { background:#6e40c9 !important; color:#fff !important; border:none !important;
                          border-radius:6px !important; font-weight:600; padding:6px 16px; cursor:pointer; }
  .widget-button button:hover { background:#8957e5 !important; }
  .info-box { background:#161b22; border:1px solid #30363d; border-radius:8px;
              padding:10px 14px; font-family:monospace; font-size:12px; color:#8b949e; }
</style>
"""))
print("UI styles loaded.")

In [ ]:
# ── Load replay buffer ──────────────────────────────────────────────────────
BUFFER_PATH = "checkpoints/replay_buffer.buffer"

if not os.path.exists(BUFFER_PATH):
    candidates = glob.glob("**/*.buffer", recursive=True)
    print("replay_buffer.buffer not found. Candidates:", candidates)
    raise FileNotFoundError(f"Could not find {BUFFER_PATH}")

print(f"Loading replay buffer from: {BUFFER_PATH}  ({os.path.getsize(BUFFER_PATH)/1e6:.1f} MB)")
ckpt = torch.load(BUFFER_PATH, map_location="cpu")

obs   = ckpt["observations"]   # (N, 3, 64, 64)  uint8
rams  = ckpt["rams"]            # (N, ram_dim)
acts  = ckpt["actions"]         # (N, action_dim)
rews  = ckpt["rewards"]         # (N, 1)
buf_index    = ckpt["index"]
buf_full     = ckpt["full"]
buf_capacity = ckpt.get("capacity", obs.shape[0])

N = obs.shape[0]
ACTION_NAMES = ["UP", "DOWN", "LEFT", "RIGHT", "A", "B"]

print(f"Frames loaded : {N:,}")
print(f"Capacity      : {buf_capacity:,}")
print(f"Buffer full   : {buf_full}   |  write-head at index {buf_index}")
print(f"Obs shape     : {tuple(obs.shape)}")
print(f"RAM shape     : {tuple(rams.shape)}")
print(f"Action shape  : {tuple(acts.shape)}")
print(f"Reward range  : [{rews.min().item():.3f}, {rews.max().item():.3f}]")

In [ ]:
# ── Helper functions ────────────────────────────────────────────────────────
reward_np = rews[:, 0].numpy().astype(np.float32)

def get_frame_rgb(idx):
    """Return (H, W, 3) float32 [0,1] for frame idx."""
    return obs[idx].permute(1, 2, 0).numpy().astype(np.float32) / 255.0

def get_action_str(idx):
    a = acts[idx]
    ai = int(torch.argmax(a).item())
    return ACTION_NAMES[ai] if ai < len(ACTION_NAMES) else str(ai)

print("Helper functions ready.")

In [ ]:
# ── Build the interactive viewer ───────────────────────────────────────────
%matplotlib inline
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#0d1117',
    'text.color':       '#c9d1d9',
    'axes.edgecolor':   '#30363d',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
})

# ── Widgets ────────────────────────────────────────────────────────────────
slider = widgets.IntSlider(
    value=0, min=0, max=N-1, step=1,
    description='Frame',
    continuous_update=True,
    layout=widgets.Layout(width='95%'),
    style={'description_width': '60px'}
)

step_input = widgets.BoundedIntText(
    value=1, min=1, max=500,
    description='Step:',
    layout=widgets.Layout(width='130px'),
    style={'description_width': '40px'}
)

btn_prev   = widgets.Button(description='◀ Prev',  layout=widgets.Layout(width='90px'))
btn_next   = widgets.Button(description='Next ▶',  layout=widgets.Layout(width='90px'))
btn_start  = widgets.Button(description='⏮ Start', layout=widgets.Layout(width='90px'))
btn_end    = widgets.Button(description='End ⏭',   layout=widgets.Layout(width='90px'))

info_label = widgets.HTML(value="")
out        = widgets.Output()

# ── Draw function ──────────────────────────────────────────────────────────
def draw(idx):
    with out:
        out.clear_output(wait=True)

        fig = plt.figure(figsize=(15, 7), facecolor='#0d1117')
        gs  = GridSpec(3, 3, figure=fig,
                       left=0.04, right=0.96, top=0.93, bottom=0.06,
                       hspace=0.45, wspace=0.35)

        # ── Main frame ────────────────────────────────────────────────
        ax_main = fig.add_subplot(gs[0:2, 0:2])
        frame   = get_frame_rgb(idx)
        from PIL import Image
        pil_img = Image.fromarray((frame * 255).astype(np.uint8))
        pil_img = pil_img.resize((320, 320), Image.NEAREST)
        ax_main.imshow(np.array(pil_img))
        ax_main.set_title(f'Frame  {idx:,} / {N-1:,}',
                           color='#a371f7', fontsize=14, fontweight='bold', pad=8)
        ax_main.axis('off')
        # Border
        for spine in ax_main.spines.values():
            spine.set_edgecolor('#a371f7')
            spine.set_linewidth(2)
            spine.set_visible(True)

        # ── Metadata panel ────────────────────────────────────────────
        ax_meta = fig.add_subplot(gs[0:2, 2])
        ax_meta.axis('off')
        action_str = get_action_str(idx)
        reward_val = rews[idx, 0].item()
        ram_vals   = rams[idx].numpy()

        # Write-head indicator
        is_near_head = abs(idx - buf_index) < 100
        head_marker = " ✏️" if is_near_head else ""

        meta_lines = [
            ("ACTION",  action_str,          '#f0883e'),
            ("REWARD",  f"{reward_val:+.4f}", '#3fb950'),
        ]
        for k, (label, val, col) in enumerate(meta_lines):
            ax_meta.text(0.05, 0.92 - k*0.12, label,
                         color='#8b949e', fontsize=9, transform=ax_meta.transAxes)
            ax_meta.text(0.05, 0.85 - k*0.12, val,
                         color=col, fontsize=15, fontweight='bold',
                         transform=ax_meta.transAxes)

        # Write-head proximity note
        ax_meta.text(0.05, 0.60, f"Write head: {buf_index:,}{head_marker}",
                     color='#8b949e', fontsize=8, transform=ax_meta.transAxes)
        ax_meta.text(0.05, 0.53, f"Fill: {min(buf_index, buf_capacity) if not buf_full else buf_capacity:,} / {buf_capacity:,}",
                     color='#8b949e', fontsize=8, transform=ax_meta.transAxes)

        # RAM values as a mini bar chart
        ax_ram = fig.add_axes([0.70, 0.38, 0.26, 0.22])
        ax_ram.set_facecolor('#161b22')
        colors_ram = ['#a371f7' if v >= 0 else '#f85149' for v in ram_vals]
        ax_ram.bar(range(len(ram_vals)), ram_vals, color=colors_ram, width=0.7)
        ax_ram.set_title('RAM / Milestones', color='#8b949e', fontsize=9, pad=4)
        ax_ram.set_xticks(range(len(ram_vals)))
        ax_ram.set_xticklabels([f'M{i}' for i in range(len(ram_vals))],
                               fontsize=7, color='#8b949e')
        ax_ram.tick_params(axis='y', labelsize=7)
        for sp in ax_ram.spines.values():
            sp.set_edgecolor('#30363d')

        # ── Reward timeline ───────────────────────────────────────────
        ax_rew = fig.add_subplot(gs[2, :])
        win    = 2000
        lo     = max(0, idx - win // 2)
        hi     = min(N, lo + win)
        lo     = max(0, hi - win)
        xs     = np.arange(lo, hi)
        ys     = reward_np[lo:hi]
        ax_rew.fill_between(xs, ys, alpha=0.25, color='#a371f7')
        ax_rew.plot(xs, ys, color='#a371f7', linewidth=0.6)
        ax_rew.axvline(idx, color='#58a6ff', linewidth=1.5, linestyle='--')
        # Mark write-head position if visible
        if lo <= buf_index <= hi:
            ax_rew.axvline(buf_index, color='#f0883e', linewidth=1.2,
                           linestyle=':', label='write head')
            ax_rew.legend(fontsize=7, labelcolor='#f0883e',
                          facecolor='#161b22', edgecolor='#30363d')
        ax_rew.set_xlim(lo, hi)
        ax_rew.set_title(f'Reward timeline  (window {lo:,}–{hi:,})',
                          color='#8b949e', fontsize=9, pad=4)
        ax_rew.set_xlabel('Frame index', color='#8b949e', fontsize=8)
        for sp in ax_rew.spines.values():
            sp.set_edgecolor('#30363d')

        plt.suptitle('🗂️  Replay Buffer Viewer', color='#c9d1d9',
                     fontsize=16, fontweight='bold', y=0.99)
        plt.show()

    # Update text info below
    info_label.value = (
        f'<div class="info-box">'
        f'<b style="color:#a371f7">Frame {idx:,}</b> &nbsp;|&nbsp; '
        f'Action: <b style="color:#f0883e">{get_action_str(idx)}</b> &nbsp;|&nbsp; '
        f'Reward: <b style="color:#3fb950">{rews[idx,0].item():+.4f}</b> &nbsp;|&nbsp; '
        f'Buffer fill: <b>{N:,}</b> frames &nbsp;|&nbsp; '
        f'Write-head: <b>{buf_index:,}</b>'
        f'</div>'
    )

# ── Callbacks ──────────────────────────────────────────────────────────────
def on_slider(change):
    draw(change['new'])

def on_prev(_):
    slider.value = max(0, slider.value - step_input.value)

def on_next(_):
    slider.value = min(N-1, slider.value + step_input.value)

def on_start(_):
    slider.value = 0

def on_end(_):
    slider.value = N - 1

slider.observe(on_slider, names='value')
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_start.on_click(on_start)
btn_end.on_click(on_end)

# ── Layout & display ──────────────────────────────────────────────────────
controls = widgets.HBox(
    [btn_start, btn_prev, step_input, btn_next, btn_end],
    layout=widgets.Layout(justify_content='center', margin='6px 0')
)

display(widgets.VBox([
    widgets.HTML('<div style="background:#0d1117;border-radius:10px;padding:10px 0">'
                 '<h2 style="color:#a371f7;font-family:Segoe UI,sans-serif;'
                 'text-align:center;margin:0">🗂️ Replay Buffer Viewer</h2>'
                 '<p style="color:#8b949e;text-align:center;font-size:12px;margin:4px 0">'
                 'Drag the slider · use ◀ ▶ buttons · or scroll the slider with the mouse wheel'
                 '</p></div>'),
    slider,
    controls,
    info_label,
    out,
], layout=widgets.Layout(width='100%')))

# ── Scroll-wheel support via JS ────────────────────────────────────────────
display(HTML("""
<script>
(function() {
  function attachWheel() {
    var sliders = document.querySelectorAll('.widget-hslider input[type=range]');
    sliders.forEach(function(sl) {
      if (sl._wheelAttached) return;
      sl._wheelAttached = true;
      sl.addEventListener('wheel', function(e) {
        e.preventDefault();
        var step = parseInt(document.querySelector('.widget-bounded-int-text input')?.value || '1');
        var delta = e.deltaY < 0 ? step : -step;
        var newVal = Math.min(parseInt(sl.max), Math.max(parseInt(sl.min), parseInt(sl.value) + delta));
        sl.value = newVal;
        sl.dispatchEvent(new Event('input', {bubbles: true}));
        sl.dispatchEvent(new Event('change', {bubbles: true}));
      }, {passive: false});
    });
  }
  // Retry a few times until slider is rendered
  var tries = 0;
  var iv = setInterval(function() {
    attachWheel();
    tries++;
    if (tries > 20) clearInterval(iv);
  }, 500);
})();
</script>
"""))

# Draw frame 0 immediately
draw(0)